# 01 — attribute Access, Descripthenrs and Properties (Deep Dive)**Level:** Advanced OOP · **Series:** OOP these Python → Advanceevery time when tum `obj.name` writete be, Python ek complete **algorithm** chalata is — only dictionarylookup not is. Us algorithm of dil is **descripthenr protocol**. `property`, `classmethod`,`staticmethod`, `__slots__`, functions of bound-method ban jana, ORM of `Field`s — all isi ekmechanism par khare are.These notebook it mechanism to **zero from khol do** dikhata is.## Ththis notebook in what we will learn- `obj.x` of **exact lookup order**: metaclass → `type(obj).__mro__` data descripthenrs → instance `__dict__` → class non-data descripthenrs/attrs → `__getattr__`- `__getattribute__` override then do (and **infinite recursion trap** from bachna)- `__getattr__` vs `__getattribute__` — who when runs is- `__setattr__`, `__delattr__`, `__dir__` customize then do- Descripthenr protocol: `__get__`, `__set__`, `__delete__`, `__set_name__`- **Data vs non-data descripthenr** precedence of live proof- function → method binding: methods asal in descripthenrs of wajah from banti are- `property` to **scratch from** then write; `classmethod`/`staticmethod` to descripthenrs from createna- Real descripthenrs: `TypedField`, `RangeField`, `Positive`, `NonEmpty`, `UnitField`, `AuditedField`, `LazyProperty` vs `functhenols.cached_property`- Descripthenr + `__slots__` + `dataclass` of interplay (here log phaste are)- **ORM-style declarative Model**: Field descripthenrs → validation + `then_dict()` + SQL generate- Pitfalls: descripthenr par shared class-level state, per-instance dattheek from store then do, `__set_name__` why important is- Performance considerations (plain attr vs property vs descripthenr — real timings)- **LeetCode design problems** (146, 155, 706, 380) OOP nazarithese from## Prerequisites- Classes, `self`, inheritance, MRO of basic understand- `property` of basic use (getter/setter) — if not pata, any masla no, hum here dobara createthesenge- Dunder methods concept of (`__init__`, `__repr__`)> **Padhne of method:** every code cell chala do output parho, then markdown of> "output means" hissa parho. Only theory parhne from descripthenrs understand not aate.

## 1. Pehla question: `obj.x` asal in what does?**What is:** `obj.x` ek *expression* is that Python inside `type(obj).__getattribute__(obj, "x")`in tarjuma is. Yani attribute access ek **method call** is — and isi lithese hum usecustomize can do.**why ahem is:** if tumhein these order pata is then tum bata can be of conflict of soorat inwho jeetega — instance of data or class of descripthenr. Interview in these favourite question is.### Default algorithm (`object.__getattribute__`) — steps```obj.x ==> type(obj).__getattribute__(obj, "x")1. cls = type(obj); cls.__mro__ in "x" dhoondo -> meta_attr2. if mila and it par __get__ and __set__/__delete__ both in from any is=> DATA descriptor => uska __get__(obj, cls) return do [all from ZYADA priority]3. warna obj.__dict__["x"] see => mil gaythen wahi return do (as-is)4. warna step 1 in mila hua class attribute see:- if it par only __get__ is => NON-DATA descripthenr => __get__(obj, cls)- warna plain class attribute return do5. some also not mila => AttributeError rsuch be6. AttributeError rsuch hone par HI Python cls.__getattr__(obj, "x") to bulata is (if mojood be)```**Yaad havene of jumla:** *"Data descripthenr > instance dict > non-data descripthenr / class attr >`__getattr__`."*

In [ ]:
# Chalo lookup order to print do do dekhein — now only class attr vs instance dict.class Config:
    debug = False          # class attribute (shared)

    def __init__(self, name):
        self.name = name   # instance attribute (per object, obj.__dict__ me goes is)


c = Config("prod")

print("c.__dict__          ->", c.__dict__)          # sirf instance of apna data
print("'debug' in c.__dict__ ->", "debug" in c.__dict__)  # False: class par is, instance par no
print("c.debug             ->", c.debug)             # step 4: class attribute mila
print("Config.debug        ->", Config.debug)

# Now instance par SAME naam from value writeo -> shadowing be jathesegic.debug = True                                        # instance __dict__ me naya key
print("\nbaad me c.__dict__ ->", c.__dict__)
print("c.debug             ->", c.debug, " (instance dict jeet gaya)")
print("Config.debug        ->", Config.debug, " (class that ab bhi purani)")

# Instance that delete do -> dobara class that dikhne lagegidel c.debug
print("\ndel of baad c.debug ->", c.debug, " (class attr wapis dikh raha is)")

**output means:** `c.debug = True` ne class of value **change not of** — only instance of`__dict__` in ek naya entry createya that class that to *chhupa* gives is (shadowing). `del c.debug`from that parda hat gaya.**Common mistake:** log understandte are `c.debug = True` ne `Config.debug` largel diya. no.class-level state largelne for `Config.debug = True` then write parta is (ya `type(c).debug`).**Gotcha (utable class attribute):** if class attribute list/dict be and tum `c.items.append(1)`do then any shadowing not is — tum **shared** object to mutate do rahe be, all instancesmutasir honge. Isi lithese utable defaults `__init__` in create.

In [ ]:
# utable class attribute of classic bug — dekh lo so that zindagi in know not phanso.class BadCart:
    items = []             # GALAT: all instances is EK list to share will do

    def add(self, x):
        self.items.append(x)   # koi assignment no -> shadowing no -> shared list mutate


a, b = BadCart(), BadCart()
a.add("laptop")
print("b.items (BadCart) ->", b.items, " <- b ne some add no did tha!")
print("same object?      ->", a.items is b.items)


class GoodCart:
    def __init__(self):
        self.items = []    # THEEK: every instance of apni list

    def add(self, x):
        self.items.append(x)


x, y = GoodCart(), GoodCart()
x.add("laptop")
print("\ny.items (GoodCart) ->", y.items)
print("same object?       ->", x.items is y.items)
assert x.items is not y.items

## 2. `__getattribute__` — every access of gatekeeper**What is:** `__getattribute__` **every** attribute access par runs is (mile or not mile).`__getattr__` only **fallback** is — then runs is when normal lookup `AttributeError` give.**why override do:** logging/tracing, proxy objects, access control, lazy loading.**Syntax:**```pythonclass C:def __getattribute__(self, name):# ... apna work ...return object.__getattribute__(self, name) # real work juste class from doao```### ⚠️ Infinite recursion trap`__getattribute__` inside if tum `self.some_bhi` writeo then that **dobara**`__getattribute__` to bulathesega → infinite recursion → `RecursionError`.Hal: `object.__getattribute__(self, name)` or `super().__getattribute__(name)` use do.

In [ ]:
# Pehle GALAT method (jaan boojh do) — try/except in lipta hua so that notebook runs rahe.class RecursiveTrap:
    def __init__(self):
        self.value = 10

    def __getattribute__(self, name):
# GALAT: self.log_count itself __getattribute__ to bulata is -> recursion        self.log_count = 0
        return object.__getattribute__(self, name)


try:
    RecursiveTrap().value
except RecursionError as e:
    print("RecursionError pakra gaya:", type(e).__name__)
    print("Sabaq: __getattribute__ of inside self.<attr> likhna = apne aap to bulana.")

In [ ]:
# THEEK method: every access log do, but internal work object.__getattribute__ from.class Traced:
    def __init__(self, value):
# __init__ in assignment __setattr__ from guzarta is, __getattribute__ from not        object.__setattr__(self, "_log", [])   # safest: seedha object of setattr
        object.__setattr__(self, "value", value)

    def __getattribute__(self, name):
# log list to also object.__getattribute__ from only lo (warna recursion)        log = object.__getattribute__(self, "_log")
        if not name.startswith("_"):
            log.append(name)                    # sirf public attrs trace do
        return object.__getattribute__(self, name)   # asli lookup

    def double(self):
        return self.value * 2


t = Traced(21)
print("t.value  ->", t.value)
print("t.double() ->", t.double())              # 'double' + inside 'value' both log honge
print("access log ->", object.__getattribute__(t, "_log"))
assert object.__getattribute__(t, "_log") == ["value", "double", "value"]

**output means:** log in `value`, `double`, `value` aaya — `t.double()` ne first method tolookup done, then method inside `self.value` ne dobara lookup done. Yani **method call also ekattribute access is**.**Interview tip:** "`__getattribute__` and `__getattr__` in difference?" → `__getattribute__` **hamesha**runs is and default implementation complete lookup algorithm is; `__getattr__` only **naalessi ofafter** bulaya goes. `__getattribute__` override then do khatarnak is (performance + recursion),`__getattr__` safe and sasta is.

## 3. `__getattr__` — only fallback (and bohat work of)**What is:** when normal lookup fail be jathese (AttributeError) then Python `__getattr__(self, name)`bulata is. Ismeans: **jo attributes mojood are unfor these runs only no** → zero cost.**where use is:**- Dynamic/proxy objects (`obj.anything` → API call, config key, JSON field)- Deprecated names to nathese names par redirect then do- attribute-style dictionary access (`cfg.datease.host`)**Gotcha:** `__getattr__` in *hamesha* `AttributeError` rsuch do if naam handle not can,warna `hasattr`, `copy`, `pickle` like stdlib thenols that `__deepcopy__`/`__reduce__` dhoondte are,galat jawnow pa do thenot jate are.

In [ ]:
class AttrDict:
    """dict jise obj.key of type bhi parh sakein (nested bhi)."""

    def __init__(self, data):
# __dict__ in seedha daal do -> normal lookup first isi to dhoondega        self._data = dict(data)

    def __getattr__(self, name):
# NOTE: these only then runs is when 'name' normal ways from not mile        try:
            value = self._data[name]
        except KeyError:
# LAZMI: unknown naam par AttributeError, warna hasattr/copy/pickle thenot jate are            raise AttributeError(f"{type(self).__name__!r} me koi attribute {name!r} no") from None
# nested dict to also AttrDict create do -> cfg.db.host will run        return AttrDict(value) if isinstance(value, dict) else value

    def __dir__(self):
# then completion / dir() in keys also dikhao        return sorted(set(list(super().__dir__()) + list(self._data)))


cfg = AttrDict({"env": "prod", "db": {"host": "127.0.0.1", "port": 5432}})
print("cfg.env      ->", cfg.env)
print("cfg.db.host  ->", cfg.db.host)      # nested
print("cfg.db.port  ->", cfg.db.port)

try:
    cfg.missing
except AttributeError as e:
    print("missing par  ->", e)

print("hasattr(cfg,'env')     ->", hasattr(cfg, "env"))
print("hasattr(cfg,'nope')    ->", hasattr(cfg, "nope"))   # AttributeError -> False
print("dir() me keys dikh rahi:", [n for n in dir(cfg) if not n.startswith("_")])
assert cfg.db.port == 5432 and not hasattr(cfg, "nope")

In [ ]:
# Use-case 2: purane attribute names to deprecate then do, without code thenray.import warnings

class ApiClient:
    _renamed = {"baseUrl": "base_url", "apiKey": "api_key"}   # purana -> naya

    def __init__(self, base_url, api_key):
        self.base_url = base_url
        self.api_key = api_key

    def __getattr__(self, name):
# only them naamon par will run that mojood not (yani purane camelCase names)        if name in self._renamed:
            new = self._renamed[name]
            warnings.warn(f"{name!r} deprecated is, {new!r} use do", DeprecationWarning, stacklevel=2)
            return getattr(self, new)          # naye naam par redirect
        raise AttributeError(name)


client = ApiClient("https://api.test", "k-123")
# catch_warnings(record=True) from warning capture do (stderr ganda not is)with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    old = client.baseUrl                       # purana naam
print("purane naam from mila ->", old)
print("warning category    ->", caught[0].category.__name__)
print("warning message     ->", caught[0].message)
assert old == "https://api.test" and caught[0].category is DeprecationWarning

## 4. `__setattr__`, `__delattr__`, `__dir__`- `__setattr__(self, name, value)` — **every** assignment (`obj.x = 1`) par runs is. Here alsorecursion trap: inside `self.x = ...` writene from dobara itself bulao ge. `object.__setattr__` or`self.__dict__[name] = value` use do.- `__delattr__(self, name)` — `del obj.x` par.- `__dir__(self)` — `dir(obj)` of output.**Use-cases:** read-only / frozen objects, allowed-fields whitelist, authen type-coercion,change tracking (dirty flags for ORM).

In [ ]:
class Frozen:
    """Ek immutable-ish object: banne of baad koi attribute set/delete no be can."""

    _ready = False   # class-level flag: __init__ of doran False rehta is

    def __init__(self, **kw):
        for k, v in kw.items():
            self.__dict__[k] = v        # __setattr__ bypass (seedha dict me)
# instance par flag set do — now from lock        self.__dict__["_ready"] = True

    def __setattr__(self, name, value):
        if self._ready:                 # normal lookup: instance dict -> True
            raise AttributeError(f"{type(self).__name__} immutable is; {name!r} set no be can")
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError(f"{type(self).__name__} immutable is; {name!r} delete no be can")

    def __repr__(self):
        inner = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items() if k != "_ready")
        return f"Frozen({inner})"


p = Frozen(x=3, y=4)
print(p)
for action, fn in [("set", lambda: setattr(p, "x", 99)), ("del", lambda: delattr(p, "x"))]:
    try:
        fn()
    except AttributeError as e:
        print(f"{action} rok diya gaya ->", e)
assert p.x == 3

In [ ]:
class StrictModel:
    """Sirf declared fields allowed — typo par foran error (dirty tracking of sath)."""

    ALLOWED = {"id", "title", "price"}

    def __init__(self, **kw):
        object.__setattr__(self, "_dirty", set())   # who who from field badle
        for k, v in kw.items():
            setattr(self, k, v)                     # apna __setattr__ hi runs (validation of for)
        self._dirty.clear()                         # initial load to "change" na ginein

    def __setattr__(self, name, value):
        if name not in self.ALLOWED and not name.startswith("_"):
# typo pakro: obj.titel = "x" chup chaap not runs            raise AttributeError(f"unknown field {name!r}; allowed: {sorted(self.ALLOWED)}")
        object.__setattr__(self, name, value)
        if not name.startswith("_"):
            self._dirty.add(name)                   # change track do

    def changed(self):
        return sorted(self._dirty)


m = StrictModel(id=1, title="Book", price=250)
print("shuru me changed ->", m.changed())
m.price = 300
print("price badalne of baad ->", m.changed())
try:
    m.titel = "typo!"                               # jaan boojh kar galat naam
except AttributeError as e:
    print("typo pakra gaya ->", e)
assert m.changed() == ["price"]

## 5. Descripthenr protocol — real engine**What is:** any also **class** that these in from any method define doe, that **descripthenr** is:| method | Know runs is | Signature ||---|---|---|| `__get__(self, obj, objtype=None)` | value parhte time | `obj` = instance (ya `None` if class from access) || `__set__(self, obj, value)` | value writete time | assignment intercept || `__delete__(self, obj)` | `del obj.attr` | deletion intercept || `__set_name__(self, owner, name)` | class banne of time **ek dafa** | apna naam pata runs is |**Do qismein:**- **Data descripthenr** — `__set__` or `__delete__` (less az less ek) mojood be. Instance `__dict__`from **more** priority.- **Non-data descripthenr** — only `__get__`. Instance `__dict__` from **less** priority.**Snow from ahem thing:** descripthenr **class of attribute** of thenr par havea goes, instance parno. `self.field = TypedField()` work **no** doega.```pythonclass Model:age = TypedField(int) # <- here (class body me) only work does```

In [ ]:
# Snow from first, all from saada descripthenr: every call to print does (protocol dekhne for).class LoudDescriptor:
    def __set_name__(self, owner, name):
# class banne of time Python itself bulata is: owner=class, name=attribute of naam        print(f"  __set_name__(owner={owner.__name__}, name={name!r})")
        self.name = name
        self.storage = "_" + name            # per-instance data yahan rakhenge

    def __get__(self, obj, objtype=None):
        if obj is None:
            print(f"  __get__ class from ({objtype.__name__}.{self.name}) -> khud descriptor")
            return self                       # Model.age -> descriptor object
        print(f"  __get__ instance from ({self.name})")
        return getattr(obj, self.storage, None)

    def __set__(self, obj, value):
        print(f"  __set__({self.name}, {value!r})")
        setattr(obj, self.storage, value)     # instance of apne dict me

    def __delete__(self, obj):
        print(f"  __delete__({self.name})")
        delattr(obj, self.storage)


print("class define be rahi is:")
class Demo:
    x = LoudDescriptor()      # <- __set_name__ abhi will run

print("\nobject create:")
d = Demo()
print("\nlikhna: d.x = 5")
d.x = 5
print("\nparhna: d.x")
print("  natija ->", d.x)
print("\nclass from: Demo.x")
print("  natija ->", type(Demo.x).__name__)
print("\ndelete: del d.x")
del d.x
print("\nab d.x ->", d.x, "(storage gayab, default None)")

**output means:**1. `__set_name__` **class body chalte time** only chal gaya — object createne from first. Isi lithesedescripthenr to apna naam itself pata chal goes.2. `obj is None` check important is: `Demo.x` (class from access) in `obj=None` comes. Isi wajah from`property` also class from access doing par `property` object gives is, value no.3. Data we have `obj.__dict__["_x"]` in havea — **descripthenr par no**. These all from important niyamis (next section iski wajah shows).

## 6. ☠️ Pitfall #1: descripthenr par per-instance datthen have**Ghalti:** `self.value = value` descripthenr inside write to give. But descripthenr object **the same**is poori class for — all instances the same to share do. Natija: ek object of valuedoosre in dikhne feels.**Hal (3 ways):**1. `obj.__dict__[self.sthenrage]` — all from common and tez (per-instance)2. `WeakKeyDictionary` — when owner class in `__slots__` be or `__dict__` chhedna not be3. Naming: `self.sthenrage = "_" + name` (`__set_name__` from), or private mangled naam

In [ ]:
# GALAT: shared state — do objects a fewosre of data khich rahe are.class BrokenField:
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return self.value                    # BUG: descriptor par ek hi 'value'

    def __set__(self, obj, value):
        self.value = value                   # BUG: all instances yahi likh rahe are


class BrokenUser:
    name = BrokenField()


u1, u2 = BrokenUser(), BrokenUser()
u1.name = "Ali"
u2.name = "Sara"
print("u1.name ->", u1.name, " <- 'Ali' hona should tha!")
print("u2.name ->", u2.name)
assert u1.name == "Sara", "yahi bug is: shared state"
print("Sabaq: descriptor object PER-class is, per-instance no.")

In [ ]:
# THEEK #1: instance of __dict__ in store do.class Field:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage = f"_{name}_value"      # collision from bachne of for

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        try:
            return obj.__dict__[self.storage]
        except KeyError:
            raise AttributeError(f"{self.name!r} abhi set no hua") from None

    def __set__(self, obj, value):
        obj.__dict__[self.storage] = value   # PER-INSTANCE


class User:
    name = Field()
    city = Field()


a, b = User(), User()
a.name, a.city = "Ali", "Lahore"
b.name, b.city = "Sara", "Karachi"
print("a ->", a.name, a.city)
print("b ->", b.name, b.city)
print("a.__dict__ ->", a.__dict__)
assert (a.name, b.name) == ("Ali", "Sara")

fresh = User()
try:
    fresh.name
except AttributeError as e:
    print("unset par ->", e)

In [ ]:
# THEEK #2: WeakKeyDictionary — when owner class in __slots__ be (any __dict__ no).import weakref

class WeakField:
    def __init__(self):
        self._values = weakref.WeakKeyDictionary()   # object mar jaye to entry bhi mit jaye

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return self._values.get(obj)

    def __set__(self, obj, value):
        self._values[obj] = value


class Point:
    __slots__ = ("__weakref__",)    # __dict__ no, magar weakref support should
    x = WeakField()
    y = WeakField()


p = Point()
p.x, p.y = 3, 4
print("p.x, p.y ->", p.x, p.y)
print("has __dict__? ->", hasattr(p, "__dict__"))
print("live entries ->", len(Point.x._values))
del p
import gc; gc.collect()
print("delete of baad entries ->", len(Point.x._values), "(memory leak no hui)")
assert len(Point.x._values) == 0

**output means:** `WeakKeyDictionary` ne object marne par entry itself saaf do di. if normal`dict` use do then descripthenr humesha reference pakre keeps → **memory leak**, object knowcollect not is.**Trade-off:** `WeakKeyDictionary` for key **hashnowle** and weak-referencenowle honi needed.`__slots__` that class in `"__weakref__"` add then do parta is. Normal case in `obj.__dict__`method only behtar (tez + simple) is.

## 7. `__set_name__` why important is it?Python 3.6 from first descripthenr to apna naam **manually** batana parta tha:```pythonclass Model:age = TypedField(int, name="age") # naam do time then write para -> DRY violation + typo risk````__set_name__(self, owner, name)` class creation of time authenmatic runs is and descripthenr touska attribute naam give gives is.**Know no runs (gotcha):** if descripthenr class body of outside attach do:```pythonclass C: passC.x = Field()  # __set_name__ no will run! itself bulao:Field.__set_name__(C.x, C, "x") # or type(C.x).__set_name__(C.x, C, "x")```

In [ ]:
class NeedsName:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(f"_{self.name}")

    def __set__(self, obj, value):
        obj.__dict__[f"_{self.name}"] = value


# Case A: class body in -> __set_name__ chalaclass InBody:
    field = NeedsName()

print("class body me   -> name =", InBody.field.name)

# Case B: later attach -> __set_name__ no chalaclass Later:
    pass

Later.field = NeedsName()
print("baad me attach  -> name attribute is?", hasattr(Later.field, "name"))
try:
    Later().field
except AttributeError as e:
    print("istemal par crash ->", type(e).__name__, "-", e)

# Manually bulao -> theek be gayatype(Later.field).__set_name__(Later.field, Later, "field")
obj = Later()
obj.field = 7
print("manual __set_name__ of baad ->", obj.field)
assert obj.field == 7 and InBody.field.name == "field"

## 8. Data vs non-data descripthenr — precedence of live proofThisi that hissa is that interview in all from more poocha goes. Chalo **sanoit** do:- **Data descripthenr** (`__get__` + `__set__`): instance `__dict__` in same naam be **then bhi**descripthenr jeetega.- **Non-data descripthenr** (only `__get__`): instance `__dict__` jeetega.Thisi wajah is of `functhenols.cached_property` (non-data) pehli time compute do of value to`obj.__dict__` in daal deti is and agli time **descripthenr absolutely runs only no** → zero overhead.and `property` (data descripthenr) hamesha chalti is.

In [ ]:
class DataDesc:
    """__get__ + __set__ => DATA descriptor."""
    def __get__(self, obj, objtype=None):
        return "DATA-descriptor of value"
    def __set__(self, obj, value):
        print("   (DataDesc.__set__ chala — value instance dict me no gayi)")


class NonDataDesc:
    """sirf __get__ => NON-DATA descriptor."""
    def __get__(self, obj, objtype=None):
        return "NON-DATA-descriptor of value"


class Battle:
    d = DataDesc()
    n = NonDataDesc()


b = Battle()
print("khali instance dict:", b.__dict__)
print("b.d ->", b.d)
print("b.n ->", b.n)

# Now instance __dict__ in ZABARDASTI same naam daalo (__set__ bypass do hue)b.__dict__["d"] = "instance dict of value"
b.__dict__["n"] = "instance dict of value"
print("\ninstance dict bhar diya:", b.__dict__)
print("b.d ->", b.d, "  <== DATA descriptor JEETA (dict ignore hua)")
print("b.n ->", b.n, "  <== instance dict JEETA (descriptor ignore hua)")

assert b.d == "DATA-descriptor of value"
assert b.n == "instance dict of value"
print("\nNiyam saabit: data descriptor > instance __dict__ > non-data descriptor")

In [ ]:
# Practical asar: itself of LazyProperty (non-data) vs property (data) — kitni time compute is.calls = {"lazy": 0, "prop": 0}

class LazyProperty:
    """Non-data descriptor: pehli dafa compute, phir instance dict me cache -> agli dafa bypass."""

    def __init__(self, func):
        self.func = func
        self.__doc__ = func.__doc__

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        value = self.func(obj)
        obj.__dict__[self.name] = value    # SAME naam from dict me -> non-data hone of wajah from jeet
        return value


class Report:
    def __init__(self, rows):
        self.rows = rows

    @LazyProperty
    def total_lazy(self):
        """Mehngi calculation (cache hone of qabil)."""
        calls["lazy"] += 1
        return sum(self.rows)

    @property
    def total_prop(self):
        calls["prop"] += 1
        return sum(self.rows)


r = Report([1, 2, 3, 4, 5])
for _ in range(3):
    r.total_lazy, r.total_prop

print("total_lazy value ->", r.total_lazy, "| compute hua:", calls["lazy"], "dafa")
print("total_prop value ->", r.total_prop, "| compute hua:", calls["prop"], "dafa")
print("instance dict     ->", {k: v for k, v in r.__dict__.items() if k != 'rows'})
assert calls["lazy"] == 1, "lazy sirf ek dafa"
assert calls["prop"] == 4, "property every dafa"

In [ ]:
# functhenols.cached_property wahi work doti is — and stale-cache of masla also wahi is.from functools import cached_property

class Basket:
    def __init__(self, prices):
        self.prices = list(prices)
        self.computed = 0

    @cached_property
    def total(self):
        self.computed += 1
        return sum(self.prices)


bk = Basket([100, 200])
print("total       ->", bk.total, "| computed:", bk.computed)
bk.prices.append(700)                    # data badal gaya...
print("data badla, total ->", bk.total, "| computed:", bk.computed, " <- STALE!")

# Cache invalidate doing of method: dict from key hata dodel bk.__dict__["total"]                 # ya: bk.__dict__.pop("total", None)
print("cache clear of baad ->", bk.total, "| computed:", bk.computed)
assert bk.total == 1000 and bk.computed == 2
print("\nSabaq: cached_property sirf un values par jo instance of lifetime me no badalti.")

## 9. function → method: binding of raaz**Sawal:** `obj.method()` in `self` **who** gives is it?**Jawnow:** functions also **non-data descripthenrs** are! `function.__get__(obj, cls)` ek**bound method** object creates that `obj` to first argument of way pakre has.```class.method  -> plain function  (obj=None mila, function itself return hua)obj.method  -> bound method   (function.__get__(obj, class))obj.method(1)  -> function(obj, 1)  -> self=obj```Isi lithese `self` "magic" not is — these descripthenr protocol of natural natija is.

In [ ]:
class Greeter:
    def hello(self, who):
        return f"Hello {who}, main {type(self).__name__} hoon"


g = Greeter()

print("type(Greeter.hello)  ->", type(Greeter.hello).__name__)   # function
print("type(g.hello)        ->", type(g.hello).__name__)         # method (bound)
print("g.hello.__self__ is g ->", g.hello.__self__ is g)         # bound object
print("g.hello.__func__ is Greeter.hello ->", g.hello.__func__ is Greeter.hello)

# Manually bind do — absolutely wahi natijamanual = Greeter.hello.__get__(g, Greeter)
print("\nmanual bind ->", manual("Ali"))
print("normal      ->", g.hello("Ali"))
assert manual("Ali") == g.hello("Ali")

# Unbound function to self itself to give paraprint("\nfunction to direct bulao ->", Greeter.hello(g, "Sara"))

# Functions par __get__ is but __set__ not => NON-DATA descripthenrprint("\nfunction par __get__? ->", hasattr(Greeter.hello, "__get__"))
print("function par __set__? ->", hasattr(Greeter.hello, "__set__"))

In [ ]:
# Non-data hone of asal asar: instance par same naam have do then method 'shadow' be goes.class Calc:
    def double(self, x):
        return x * 2


c = Calc()
print("class of method ->", c.double(5))

c.double = lambda x: x * 100          # instance __dict__ me plain function (bind no will be)
print("instance ne shadow did ->", c.double(5), " <- self no mila, x=5")
print("c.__dict__ ->", list(c.__dict__))

del c.double
print("shadow hatane of baad ->", c.double(5))
assert c.double(5) == 10

# GOTCHA: instance par havea function bind not is — isi lithese monkey-patch class par doCalc.triple = lambda self, x: x * 3    # class par -> bind will be
print("\nclass par patch ->", c.triple(5))

**Interview gold:** "`staticmethod` why needed when hum instance par function have can are?"→ Kyunki **class** par havea function bind is (`self` chala goes). `staticmethod` bindingto rok gives is. and instance par havea function bind not is but that every object for separatememory takes is and `pickle`/`copy` in masle gives is.

## 10. `property` to scratch from create`property` any language keyword not — these ek **data descripthenr class** is. Chalo apni`MyProperty` writeein: `__get__`, `__set__`, `__delete__`, and `.setter` / `.deleter` chaining.

In [ ]:
class MyProperty:
    """builtin property of apna version — data descriptor."""

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget, self.fset, self.fdel = fget, fset, fdel
# docstring getter from lo (help() for)        self.__doc__ = doc if doc is not None else getattr(fget, "__doc__", None)

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self                    # class.attr -> descriptor khud
        if self.fget is None:
            raise AttributeError(f"{self.name!r} readable no (koi getter no)")
        return self.fget(obj)

    def __set__(self, obj, value):         # <- iski mojudgi isay DATA descriptor banati is
        if self.fset is None:
            raise AttributeError(f"{self.name!r} read-only is")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError(f"{self.name!r} delete no be can")
        self.fdel(obj)

# --- decorator chaining: @x.setter naya object creates (imutable style) ---    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)


class Circle:
    def __init__(self, radius):
        self.radius = radius               # setter from guzrega (validation of sath)

    @MyProperty
    def radius(self):
        """Circle of radius (hamesha > 0)."""
        return self._radius

    @radius.setter
    def radius(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError("radius number hona should")
        if value <= 0:
            raise ValueError(f"radius > 0 hona should, mila {value}")
        self._radius = value

    @MyProperty
    def area(self):
        """Computed, read-only."""
        return 3.141592653589793 * self._radius ** 2


ci = Circle(2)
print("radius ->", ci.radius)
print("area   ->", round(ci.area, 4))
print("docstring ->", Circle.radius.__doc__)

ci.radius = 5
print("naya area ->", round(ci.area, 4))

for bad, exc in [(-1, ValueError), ("abc", TypeError)]:
    try:
        ci.radius = bad
    except exc as e:
        print(f"{bad!r} rok diya ->", type(e).__name__, "-", e)

try:
    ci.area = 100                          # koi setter no
except AttributeError as e:
    print("read-only ->", e)

assert ci.radius == 5
assert isinstance(Circle.radius, MyProperty)

In [ ]:
# Sanity check: builtin property from behaviour match does?class WithBuiltin:
    def __init__(self, v):
        self._v = v

    @property
    def v(self):
        return self._v * 2

    @v.setter
    def v(self, x):
        self._v = x


class WithMine:
    def __init__(self, v):
        self._v = v

    @MyProperty
    def v(self):
        return self._v * 2

    @v.setter
    def v(self, x):
        self._v = x


bi, mi = WithBuiltin(10), WithMine(10)
print("builtin ->", bi.v, "| mine ->", mi.v)
bi.v = mi.v = 3
print("set of baad: builtin ->", bi.v, "| mine ->", mi.v)
# both data descripthenrs areprint("\nbuiltin property data descriptor? ->",
      hasattr(property, "__get__") and hasattr(property, "__set__"))
print("MyProperty data descriptor?       ->",
      hasattr(MyProperty, "__get__") and hasattr(MyProperty, "__set__"))
assert bi.v == mi.v == 6

## 11. `classmethod` and `staticmethod` to descripthenrs from create- `staticmethod` — **binding rok** gives is: `__get__` only function wapis do gives is.- `classmethod` — function to **class** from bind does (`cls` first argument).both **non-data** descripthenrs are (only `__get__`).

In [ ]:
import functools

class MyStaticMethod:
    """__get__ bas function lauta gives is -> koi self/cls no."""

    def __init__(self, func):
        self.func = func
        functools.update_wrapper(self, func)   # __name__, __doc__ copy

    def __get__(self, obj, objtype=None):
        return self.func                       # <-- binding no

    def __call__(self, *a, **kw):              # class body of inside direct call of for
        return self.func(*a, **kw)


class MyClassMethod:
    """__get__ function to class from bind does is."""

    def __init__(self, func):
        self.func = func
        functools.update_wrapper(self, func)

    def __get__(self, obj, objtype=None):
        if objtype is None:
            objtype = type(obj)
# MethodType(func, objtype) => first arg cls ban goes        return functools.partial(self.func, objtype)


class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @MyClassMethod
    def from_fahrenheit(cls, f):
        """Alternative constructor — cls automatic mila."""
        return cls((f - 32) * 5 / 9)

    @MyStaticMethod
    def is_valid(value):
        """Koi self/cls no should — pure function."""
        return -273.15 <= value <= 1000

    def __repr__(self):
        return f"Temperature({self.celsius:.2f})"


t = Temperature.from_fahrenheit(212)
print("from_fahrenheit(212) ->", t)
print("instance from bhi      ->", Temperature(0).from_fahrenheit(32))
print("is_valid(25)   ->", Temperature.is_valid(25))
print("is_valid(-500) ->", Temperature.is_valid(-500))
print("instance from static ->", t.is_valid(50))
assert abs(t.celsius - 100) < 1e-9 and Temperature.is_valid(25) is True

# Inheritance with: cls SUBCLASS is (thisi classmethod of faida is)class KelvinTemp(Temperature):
    def __repr__(self):
        return f"KelvinTemp({self.celsius + 273.15:.2f}K)"


k = KelvinTemp.from_fahrenheit(32)
print("\nsubclass from ->", k, "| type:", type(k).__name__)
assert isinstance(k, KelvinTemp), "classmethod ne subclass hi banayi"

In [ ]:
# Builtin classmethod/staticmethod also descripthenrs only are — proof.class Probe:
    @classmethod
    def cm(cls): ...
    @staticmethod
    def sm(): ...
    @property
    def pr(self): ...
    def plain(self): ...


raw = {name: Probe.__dict__[name] for name in ("cm", "sm", "pr", "plain")}
print(f"{'attribute':10} {'type':18} {'__get__':>8} {'__set__':>8}  qism")
print("-" * 62)
for name, val in raw.items():
    has_get, has_set = hasattr(type(val), "__get__"), hasattr(type(val), "__set__")
    if has_get and has_set:
        kind = "DATA descriptor"
    elif has_get:
        kind = "non-data descriptor"
    else:
        kind = "plain object"
    print(f"{name:10} {type(val).__name__:18} {str(has_get):>8} {str(has_set):>8}  {kind}")

assert hasattr(type(raw["pr"]), "__set__")          # property = data
assert not hasattr(type(raw["cm"]), "__set__")      # classmethod = non-data

## 12. Real-world descripthenrs — ek chis validation libraryNow theory from practice. Hum ek **reuallle** descripthenr family createthesenge:```Validated (abstract juste: sthenrage + hook)├── TypedField -> type check├── RangeField -> numeric min/max├── Positive  -> > 0├── NonEmpty  -> string/collection khali not be├── UnitField  -> input convert do of store (km -> m)└── AuditedField -> every change of log```**Design of usool:** juste class sthenrage and error-message sambhale; subclass only `validate()`(ya `transform()`) writee. Thisi **Template method** pattern is.

In [ ]:
from abc import ABC, abstractmethod

class Validated(ABC):
    """Descriptors of for base: per-instance storage + validate() hook."""

    def __set_name__(self, owner, name):
        self.name = name
        self.storage = f"__{name}"          # instance dict me is naam from

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        try:
            return obj.__dict__[self.storage]
        except KeyError:
            raise AttributeError(
                f"{objtype.__name__}.{self.name} abhi set no hua"
            ) from None

    def __set__(self, obj, value):
        value = self.transform(obj, value)   # subclass badal can is
        self.validate(obj, value)            # subclass check does is
        obj.__dict__[self.storage] = value

    def __delete__(self, obj):
        obj.__dict__.pop(self.storage, None)

    def transform(self, obj, value):
        """Default: koi tabdeeli no. Subclass override kare (e.g. unit conversion)."""
        return value

    @abstractmethod
    def validate(self, obj, value):
        """Ghalat value par TypeError/ValueError raise do."""

    def fail(self, value, why):
        raise ValueError(f"{self.name}={value!r} qabool no: {why}")


class TypedField(Validated):
    def __init__(self, expected_type, allow_none=False):
        self.expected_type, self.allow_none = expected_type, allow_none

    def validate(self, obj, value):
        if value is None and self.allow_none:
            return
        if not isinstance(value, self.expected_type):
            names = getattr(self.expected_type, "__name__", str(self.expected_type))
            raise TypeError(
                f"{self.name} to {names} should, mila {type(value).__name__}"
            )


class RangeField(Validated):
    def __init__(self, minimum=None, maximum=None):
        self.minimum, self.maximum = minimum, maximum

    def validate(self, obj, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} number hona should")
        if self.minimum is not None and value < self.minimum:
            self.fail(value, f">= {self.minimum} hona should")
        if self.maximum is not None and value > self.maximum:
            self.fail(value, f"<= {self.maximum} hona should")


class Positive(RangeField):
    def __init__(self):
        super().__init__(minimum=0)

    def validate(self, obj, value):
        super().validate(obj, value)
        if value == 0:
            self.fail(value, "strictly > 0 hona should")


class NonEmpty(Validated):
    def transform(self, obj, value):
        return value.strip() if isinstance(value, str) else value   # trim kar do

    def validate(self, obj, value):
        if not hasattr(value, "__len__"):
            raise TypeError(f"{self.name} of length honi should (str/list/dict)")
        if len(value) == 0:
            self.fail(value, "khali no be can")


print("Validation descriptors tayyar:",
      [c.__name__ for c in (TypedField, RangeField, Positive, NonEmpty)])

In [ ]:
# Chalo them ek class par lagathesein and every validater to test do.class Product:
    name = NonEmpty()
    sku = TypedField(str)
    price = Positive()
    discount = RangeField(0, 100)
    stock = TypedField(int)
    note = TypedField(str, allow_none=True)

    def __init__(self, name, sku, price, discount=0, stock=0, note=None):
        self.name, self.sku, self.price = name, sku, price
        self.discount, self.stock, self.note = discount, stock, note

    def final_price(self):
        return round(self.price * (1 - self.discount / 100), 2)

    def __repr__(self):
        return f"Product({self.name!r}, {self.sku!r}, Rs{self.price}, -{self.discount}%)"


p = Product("  Laptop  ", "SKU-1", 150000, discount=10, stock=4)
print(p)
print("name trim hua ->", repr(p.name))
print("final price   ->", p.final_price())

bad_cases = [
    ("name", "", ValueError, "khali string"),
    ("name", "   ", ValueError, "sirf spaces (trim of baad khali)"),
    ("price", -5, ValueError, "negative price"),
    ("price", 0, ValueError, "zero price"),
    ("discount", 150, ValueError, "range from bahar"),
    ("stock", "bohat", TypeError, "galat type"),
    ("sku", 123, TypeError, "int diya str of jagah"),
]
print("\n--- Validation tests ---")
for field, value, exc, why in bad_cases:
    try:
        setattr(p, field, value)
        print(f"[FAIL] {field}={value!r} pass be gaya (no hona should tha)")
    except exc as e:
        print(f"[ok] {why:32} -> {type(e).__name__}: {e}")

p.note = None            # allow_none=True
print("\nnote=None chala ->", p.note)
assert p.price == 150000 and p.name == "Laptop"

In [ ]:
# UnitField: user which unit in give, hum canonical unit in store do.class UnitField(Validated):
    """input to canonical unit me convert kar of holds is (e.g. km/cm -> meters)."""

    FACTORS = {"m": 1.0, "km": 1000.0, "cm": 0.01, "mm": 0.001, "mi": 1609.344}

    def __init__(self, canonical="m"):
        self.canonical = canonical

    def transform(self, obj, value):
# value or then number (canonical maan lo) or (number, unit) tuple or "5 km" string        if isinstance(value, tuple) and len(value) == 2:
            num, unit = value
        elif isinstance(value, str):
            parts = value.split()
            if len(parts) != 2:
                raise ValueError(f"{self.name}: format '5 km' like do, mila {value!r}")
            num, unit = float(parts[0]), parts[1]
        else:
            num, unit = value, self.canonical
        if unit not in self.FACTORS:
            raise ValueError(f"{self.name}: unknown unit {unit!r}; {sorted(self.FACTORS)}")
        return float(num) * self.FACTORS[unit] / self.FACTORS[self.canonical]

    def validate(self, obj, value):
        if value < 0:
            self.fail(value, "distance negative no be can")


class Route:
    length = UnitField("m")          # all some meters me store will be

    def __init__(self, length):
        self.length = length


for inp in [5000, (5, "km"), "5 km", (500000, "cm"), "1 mi"]:
    r = Route(inp)
    print(f"input {str(inp):16} -> stored meters: {r.length:>10.3f}")

r = Route((5, "km"))
assert r.length == 5000.0
for bad in [(-1, "km"), (5, "parsec"), "5"]:
    try:
        Route(bad)
    except (ValueError, TypeError) as e:
        print(f"[ok] {str(bad):12} -> {type(e).__name__}: {e}")

In [ ]:
# AuditedField: every thendeeli of record (which ne, what from what) — audit trails for.from datetime import datetime, timezone

class AuditedField(Validated):
    """value badalne par per-instance change log holds is."""

    LOG_ATTR = "_audit_log"

    def __init__(self, expected_type=object):
        self.expected_type = expected_type

    def validate(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} to {self.expected_type.__name__} should")

    def __set__(self, obj, value):
        old = obj.__dict__.get(self.storage, "<unset>")
        super().__set__(obj, value)                 # validate + store
        if old != value:                            # sirf asli change log do
            log = obj.__dict__.setdefault(self.LOG_ATTR, [])
            log.append({
                "field": self.name,
                "old": old,
                "new": value,
                "at": datetime(2026, 1, 1, tzinfo=timezone.utc),   # test of for fixed
            })

    @staticmethod
    def history(obj, field=None):
        log = obj.__dict__.get(AuditedField.LOG_ATTR, [])
        return [e for e in log if field is None or e["field"] == field]


class Employee:
    name = AuditedField(str)
    salary = AuditedField(int)
    title = AuditedField(str)

    def __init__(self, name, salary, title):
        self.name, self.salary, self.title = name, salary, title


e = Employee("Ali", 100000, "Engineer")
e.salary = 120000            # promotion
e.title = "Senior Engineer"
e.salary = 120000            # same value -> log no hona should
e.salary = 150000

print("Poora audit trail:")
for entry in AuditedField.history(e):
    print(f"  {entry['field']:7} {str(entry['old']):18} -> {entry['new']!r}")

print("\nSirf salary of history:", [(x["old"], x["new"]) for x in AuditedField.history(e, "salary")])
assert len(AuditedField.history(e, "salary")) == 3      # initial + 2 real changes
# thental = name(1) + salary(3) + title(2) = 6; dobara same salary set then do log not huaassert len(AuditedField.history(e)) == 6
print("\nSame value dobara set karne par log no bara — theek.")

## 13. Descripthenr + `__slots__` + `dataclass` of interplayHere teen small but tang doing that masle are:1. **`__slots__` with descripthenr:** `__slots__` itself ek descripthenr creates. if tum`__slots__ = ("x",)` writeo **and** `x = MyField()` bhi, then class body in naam **takra** jathesega(`ValueError: 'x' these __slots__ conflicts with class variable`). Hal: slot of naam separate haveo(`_x`) and descripthenr `_x` in store doe — or `WeakKeyDictionary` use do.2. **`dataclass` with descripthenr:** Python 3.10+ in descripthenr-typed fields support are, but**default** of tafseer largel goes: `field: int = MyDesc()` in `MyDesc()` **default value**ban goes, and `dataclass` `__init__` in `self.field = default` does — that descripthenr of`__set__` from guzarta is. Ismeans: default not dene par `descripthenr object` itself value bancan is. Isi lithese descripthenr of `__get__` in `obj is None` with ek "default" strategy needed.

In [ ]:
# 1) __slots__ + class variable of conflict — live proof.src = "class Bad:\n    __slots__ = ('x',)\n    x = 42\n"
try:
    exec(src)
except ValueError as e:
    print("conflict ->", e)

# Hal: slot of naam separate (_x) and descripthenr it in writee.class SlotField:
    def __set_name__(self, owner, name):
        self.name = name
        self.slot = f"_{name}"           # this slot me mojood hona should

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        try:
            return getattr(obj, self.slot)
        except AttributeError:
            raise AttributeError(f"{self.name} set no hua") from None

    def __set__(self, obj, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} number hona should")
        setattr(obj, self.slot, value)   # slot descriptor me store


class Vec:
    __slots__ = ("_x", "_y")             # descriptor of naam from ALAG
    x = SlotField()
    y = SlotField()

    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vec({self.x}, {self.y})"


v = Vec(3, 4)
print("\nslots + descriptor ->", v, "| __dict__ is?", hasattr(v, "__dict__"))
try:
    v.z = 9                              # slots of wajah from naya attr allowed no
except AttributeError as e:
    print("naya attribute ->", type(e).__name__, "-", e)
try:
    v.x = "abc"
except TypeError as e:
    print("validation      ->", e)
assert v.x == 3 and not hasattr(v, "__dict__")

In [ ]:
# 2) dataclass + descripthenr: default of behaviour see (Python 3.10+ documented).from dataclasses import dataclass, field, fields

class IntField:
    def __set_name__(self, owner, name):
        self.storage = f"_{name}"
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self          # <- dataclass isay "default" of type dekh can is
        return getattr(obj, self.storage, 0)

    def __set__(self, obj, value):
        if isinstance(value, IntField):
# dataclass ne default (itself descripthenr) pass do diya -> 0 maan lo            value = 0
        if not isinstance(value, int):
            raise TypeError(f"{self.name} int hona should, mila {type(value).__name__}")
        setattr(obj, self.storage, value)


@dataclass
class Counter:
    hits: int = IntField()       # descriptor as "default"
    misses: int = IntField()


c1 = Counter()                   # koi arg no -> descriptor khud pass hua -> 0
c2 = Counter(hits=10, misses=3)
print("c1 ->", c1.hits, c1.misses)
print("c2 ->", c2.hits, c2.misses)
print("repr ->", c2)
try:
    Counter(hits="bohat")
except TypeError as e:
    print("validation chali ->", e)
print("\ndataclass fields ->", [f.name for f in fields(Counter)])
assert (c1.hits, c2.hits) == (0, 10)
print("Sabaq: dataclass + descriptor mumkin is, magar __set__ me 'descriptor mila' case handle do.")

## 14. 🏗️ Mini-project: ORM-style declarative ModelNow all some jor do that thing create that Django/SQLAlchemy/Pydantic like libraries doti are:```pythonclass User(Model):__thenle__ = "users"id = IntegerField(primary_key=True)email = StringField(max_length=120, unique=True)age = IntegerField(min_value=0, max_value=150, null=True)```and these all authenmatic mile:- **Validation** every assignment par- `then_dict()` / `from_dict()`- `CREATE TABLE` and `INSERT` SQL **generate**- Achha `__repr__` and `__eq__`- Missing required fields of list**Design:** `Column` (descripthenr juste) + `Model.__init_subclass__` (fields collect doing for —metaclass of need no! these NB2 of thenpic is).

In [ ]:
class Column:
    """Base column descriptor: type + constraints + SQL type."""

    py_type = object
    sql_type = "TEXT"

    def __init__(self, *, primary_key=False, unique=False, null=False, default=None):
        self.primary_key, self.unique, self.null = primary_key, unique, null
        self.default = default

    def __set_name__(self, owner, name):
        self.name = name
        self.storage = f"_col_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.storage in obj.__dict__:
            return obj.__dict__[self.storage]
        if self.default is not None:
            return self.default
        if self.null:
            return None
        raise AttributeError(f"{type(obj).__name__}.{self.name} set no hua and null allowed no")

    def __set__(self, obj, value):
        if value is None:
            if not self.null:
                raise ValueError(f"{self.name} null no be can")
            obj.__dict__[self.storage] = None
            return
        value = self.coerce(value)
        self.check(value)
        obj.__dict__[self.storage] = value

    def coerce(self, value):
        if not isinstance(value, self.py_type):
            raise TypeError(
                f"{self.name} to {self.py_type.__name__} should, mila {type(value).__name__}"
            )
        return value

    def check(self, value):
        """Subclass extra constraints yahan lagaye."""

    def ddl(self):
        parts = [self.name, self.sql_type]
        if self.primary_key:
            parts.append("PRIMARY KEY")
        if self.unique and not self.primary_key:
            parts.append("UNIQUE")
        if not self.null and not self.primary_key:
            parts.append("NOT NULL")
        return " ".join(parts)


class IntegerField(Column):
    py_type = int
    sql_type = "INTEGER"

    def __init__(self, *, min_value=None, max_value=None, **kw):
        super().__init__(**kw)
        self.min_value, self.max_value = min_value, max_value

    def coerce(self, value):
        if isinstance(value, bool):     # bool int of subclass is — mana do
            raise TypeError(f"{self.name}: bool qabool no, int do")
        if isinstance(value, str) and value.lstrip("-").isdigit():
            value = int(value)          # "42" -> 42 (helpful coercion)
        return super().coerce(value)

    def check(self, value):
        if self.min_value is not None and value < self.min_value:
            raise ValueError(f"{self.name} >= {self.min_value} hona should, mila {value}")
        if self.max_value is not None and value > self.max_value:
            raise ValueError(f"{self.name} <= {self.max_value} hona should, mila {value}")


class StringField(Column):
    py_type = str
    sql_type = "VARCHAR"

    def __init__(self, *, max_length=None, choices=None, **kw):
        super().__init__(**kw)
        self.max_length, self.choices = max_length, choices

    def coerce(self, value):
        value = super().coerce(value)
        return value.strip()

    def check(self, value):
        if self.max_length is not None and len(value) > self.max_length:
            raise ValueError(f"{self.name} of length <= {self.max_length} be, mili {len(value)}")
        if self.choices is not None and value not in self.choices:
            raise ValueError(f"{self.name} in me from be: {self.choices}, mila {value!r}")

    def ddl(self):
        base = super().ddl()
        if self.max_length:
            base = base.replace("VARCHAR", f"VARCHAR({self.max_length})", 1)
        return base


class FloatField(Column):
    py_type = float
    sql_type = "REAL"

    def coerce(self, value):
        if isinstance(value, int) and not isinstance(value, bool):
            value = float(value)        # int -> float will run
        return super().coerce(value)


class BoolField(Column):
    py_type = bool
    sql_type = "BOOLEAN"


print("Columns tayyar:", [c.__name__ for c in (IntegerField, StringField, FloatField, BoolField)])

In [ ]:
class Model:
    """Base Model — __init_subclass__ from fields collect does is (metaclass of baghair)."""

    __table__ = None

    def __init_subclass__(cls, **kw):
        super().__init_subclass__(**kw)
# parent classes of fields also lo (inheritance support)        collected = {}
        for base in reversed(cls.__mro__):
            for name, val in vars(base).items():
                if isinstance(val, Column):
                    collected[name] = val
        cls._columns = collected
        if cls.__table__ is None:
            cls.__table__ = cls.__name__.lower() + "s"

    def __init__(self, **kw):
        unknown = set(kw) - set(self._columns)
        if unknown:
            raise TypeError(f"unknown field(s): {sorted(unknown)}; allowed: {sorted(self._columns)}")
        for name, value in kw.items():
            setattr(self, name, value)       # descriptor validation will run

# --- introspection ---    @classmethod
    def columns(cls):
        return dict(cls._columns)

    def missing(self):
        out = []
        for name, col in self._columns.items():
            if col.storage not in self.__dict__ and col.default is None and not col.null:
                out.append(name)
        return sorted(out)

    def is_valid(self):
        return not self.missing()

# --- serialization ---    def to_dict(self):
        data = {}
        for name, col in self._columns.items():
            try:
                data[name] = getattr(self, name)
            except AttributeError:
                data[name] = None
        return data

    @classmethod
    def from_dict(cls, data):
        known = {k: v for k, v in data.items() if k in cls._columns}
        return cls(**known)

# --- SQL ---    @classmethod
    def create_table_sql(cls):
        cols = ",\n  ".join(col.ddl() for col in cls._columns.values())
        return f"CREATE TABLE {cls.__table__} (\n  {cols}\n);"

    def insert_sql(self):
        if not self.is_valid():
            raise ValueError(f"required fields missing: {self.missing()}")
        data = {k: v for k, v in self.to_dict().items() if v is not None}
        cols = ", ".join(data)
        marks = ", ".join("?" for _ in data)
        return f"INSERT INTO {self.__table__} ({cols}) VALUES ({marks});", tuple(data.values())

    def __repr__(self):
        inner = ", ".join(f"{k}={v!r}" for k, v in self.to_dict().items() if v is not None)
        return f"{type(self).__name__}({inner})"

    def __eq__(self, other):
        return type(self) is type(other) and self.to_dict() == other.to_dict()

    def __hash__(self):
        return hash((type(self).__name__, tuple(sorted(
            (k, v) for k, v in self.to_dict().items() if not isinstance(v, (list, dict))))))


class User(Model):
    __table__ = "users"
    id = IntegerField(primary_key=True)
    email = StringField(max_length=120, unique=True)
    name = StringField(max_length=50)
    age = IntegerField(min_value=0, max_value=150, null=True)
    role = StringField(choices=("admin", "user", "guest"), default="user")
    score = FloatField(default=0.0)
    active = BoolField(default=True)


print(User.create_table_sql())
print("\nColumns:", list(User.columns()))

In [ ]:
# Model to istemal dou = User(id=1, email="  ali@test.com ", name="Ali", age=30)
print("object   ->", u)
print("email trim hua ->", repr(u.email))
print("defaults ->", "role:", u.role, "| score:", u.score, "| active:", u.active)
print("to_dict  ->", u.to_dict())
print("\nINSERT   ->")
sql, params = u.insert_sql()
print(sql, "\nparams:", params)

# Round-trip: dict -> object -> dictback = User.from_dict(u.to_dict())
print("\nround-trip equal? ->", back == u)
assert back == u

# Validation testsprint("\n--- Model validation ---")
checks = [
    (lambda: User(id=1, email="a@b.c", name="A", age=-5), ValueError, "age negative"),
    (lambda: User(id=1, email="a@b.c", name="A", role="hacker"), ValueError, "role choices from bahar"),
    (lambda: User(id=1, email="x" * 200, name="A"), ValueError, "email bohat lamba"),
    (lambda: User(id="abc", email="a@b.c", name="A"), TypeError, "id galat type"),
    (lambda: User(id=True, email="a@b.c", name="A"), TypeError, "bool as int"),
    (lambda: User(idd=1), TypeError, "unknown field (typo)"),
]
for fn, exc, why in checks:
    try:
        fn()
        print(f"[FAIL] {why} pass be gaya")
    except exc as e:
        print(f"[ok] {why:26} -> {type(e).__name__}: {str(e)[:60]}")

# Incomplete objectpartial = User(id=2)
print("\nmissing fields ->", partial.missing())
print("is_valid       ->", partial.is_valid())
try:
    partial.insert_sql()
except ValueError as e:
    print("insert roka    ->", e)
assert partial.missing() == ["email", "name"]

# Coercion: "42" -> 42, int -> floatu2 = User(id="42", email="b@c.d", name="Sara", score=95)
print("\ncoercion -> id:", repr(u2.id), "| score:", repr(u2.score))
assert u2.id == 42 and u2.score == 95.0

In [ ]:
# Inheritance: subclass parent of columns also patheseclass AdminUser(User):
    __table__ = "admin_users"
    level = IntegerField(min_value=1, max_value=10)
    department = StringField(max_length=40, default="IT")


print("AdminUser columns ->", list(AdminUser.columns()))
print("\n" + AdminUser.create_table_sql())

adm = AdminUser(id=9, email="root@test.com", name="Root", level=10)
print("\n", adm)
print("department default ->", adm.department)
try:
    adm.level = 99
except ValueError as e:
    print("level validation ->", e)

assert "email" in AdminUser.columns() and "level" in AdminUser.columns()
assert len(AdminUser.columns()) == len(User.columns()) + 2
print("\nInheritance theek: parent + child both of columns mile.")

## 15. Performance considerationsDescripthenrs sasta abstraction is but **muft no**. every access ek Python-level function call is.**Rough tarteeb (tez from sust):**```__slots__ attr < instance __dict__ attr < cached_property (2nd+ access)< property ≈ custhenm descripthenr < __getattr__ fallback< __getattribute__ override (all from mehnga)```**Know fido do:** only **hot loops** in (lakhon accesses). Warna clarity + validation offaida overhead from more is.**Optimisation tricks:**1. Cache-nowle values par `cached_property` (non-data → 2nd access par descripthenr absolutely skip)2. Validation only `__init__`/setter me, getter in not3. Hot path in local variable in nikaal lo: `r = self.radius` then loop in `r` use do4. `__slots__` from memory + access both behtar

In [ ]:
import timeit

class Plain:
    def __init__(self, v):
        self.v = v


class Slotted:
    __slots__ = ("v",)
    def __init__(self, v):
        self.v = v


class WithProperty:
    def __init__(self, v):
        self._v = v
    @property
    def v(self):
        return self._v


class SimpleDesc:
    def __set_name__(self, owner, name):
        self.storage = f"_{name}"
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__[self.storage]
    def __set__(self, obj, value):
        obj.__dict__[self.storage] = value


class WithDescriptor:
    v = SimpleDesc()
    def __init__(self, v):
        self.v = v


class WithGetattr:
    def __init__(self, v):
        self._data = {"v": v}
    def __getattr__(self, name):
        try:
            return self._data[name]
        except KeyError:
            raise AttributeError(name) from None


N = 200_000
objs = {
    "plain __dict__": Plain(1),
    "__slots__": Slotted(1),
    "property": WithProperty(1),
    "custom descriptor": WithDescriptor(1),
    "__getattr__ fallback": WithGetattr(1),
}
results = {}
for label, obj in objs.items():
    t = timeit.timeit("obj.v", globals={"obj": obj}, number=N)
    results[label] = t

base = results["plain __dict__"]
print(f"{N:,} attribute reads:\n")
print(f"{'method':22} {'waqt (s)':>10} {'nisbat':>8}")
print("-" * 44)
for label, t in sorted(results.items(), key=lambda kv: kv[1]):
    print(f"{label:22} {t:>10.4f} {t / base:>7.2f}x")
print("\nNote: absolute numbers machine par depend do are; TARTEEB ahem is.")
assert results["plain __dict__"] < results["property"] * 2

In [ ]:
# cached_property of faida hot loop in (only dobara access par).import timeit
from functools import cached_property

class Heavy:
    def __init__(self, n):
        self.n = n

    @property
    def prop_sum(self):
        return sum(range(self.n))         # every dafa dobara

    @cached_property
    def cached_sum(self):
        return sum(range(self.n))         # sirf pehli dafa


h = Heavy(500)
h.cached_sum                              # cache garam do
t_prop = timeit.timeit("h.prop_sum", globals={"h": h}, number=5_000)
t_cache = timeit.timeit("h.cached_sum", globals={"h": h}, number=5_000)
print(f"property     : {t_prop:.4f}s")
print(f"cached_prop  : {t_cache:.4f}s")
print(f"faida        : {t_prop / t_cache:.1f}x tez")
assert t_cache < t_prop

# Trick 3: hot loop in local variableclass Point3:
    def __init__(self, x):
        self._x = x
    @property
    def x(self):
        return self._x

p = Point3(2)
t_attr = timeit.timeit("total = 0\nfor _ in range(100): total += p.x",
                       globals={"p": p}, number=2_000)
t_local = timeit.timeit("x = p.x\ntotal = 0\nfor _ in range(100): total += x",
                        globals={"p": p}, number=2_000)
print(f"\nloop me p.x    : {t_attr:.4f}s")
print(f"local me nikala: {t_local:.4f}s  ({t_attr / t_local:.1f}x tez)")

## 16. Pitfalls & Gotchas — ek jaga par| # | Pitfall | why is | Hal ||---|---|---|---|| 1 | Descripthenr par `self.value = v` | descripthenr **per-class** is, per-instance not | `obj.__dict__[self.sthenrage]` or `WeakKeyDictionary` || 2 | `self.field = Field()` `__init__` in | descripthenrs only **class** attributes work on do | class body in haveo || 3 | `__getattribute__` in `self.x` | dobara itself to call | `object.__getattribute__(self, "x")` || 4 | `__setattr__` in `self.x = v` | dobara itself to call | `object.__setattr__` or `self.__dict__[...]` || 5 | `__getattr__` in `AttributeError` not rsuch then do | `hasattr`/`copy`/`pickle` thenot jate are | unknown naam par `rsuch AttributeError(name)` || 6 | Baad in `C.x = Field()` | `__set_name__` not runs | manually bulao or class body in haveo || 7 | `cached_property` largelte data par | non-data descripthenr, dobara compute not does | `del obj.__dict__["name"]` from invalidate || 8 | utable class attribute (`items = []`) | all instances share do | `__init__` in create || 9 | `__slots__` + same naam of class variable | slot descripthenr takrata is | naam separate haveo (`_x`) || 10 | `property` to class from access do of value of umeed | `obj is None` → descripthenr itself milta is | `class.attr.fget(obj)` or instance use do || 11 | Descripthenr in `objtype`/`obj` ignore then do | `class.attr` par crash | hamesha `if obj is None: return self` || 12 | `bool` to `int` understandna validation in | `bool` `int` of subclass is | `isinstance(v, bool)` first check do |

In [ ]:
# Pitfall #10 of live demo + sahi methodclass Celsius:
    def __init__(self, c):
        self._c = c

    @property
    def fahrenheit(self):
        return self._c * 9 / 5 + 32


t = Celsius(100)
print("instance from     ->", t.fahrenheit)
print("class from        ->", type(Celsius.fahrenheit).__name__, "(value no, property object)")
print("getter manually ->", Celsius.fahrenheit.fget(t))
assert Celsius.fahrenheit.fget(t) == t.fahrenheit

# Pitfall #11: obj is None check bhula do then what it isclass Careless:
    def __get__(self, obj, objtype=None):
        return obj.__dict__["_v"]        # obj None be to AttributeError/TypeError


class Holder:
    v = Careless()


try:
    Holder.v                             # class from access -> obj=None
except (AttributeError, TypeError) as e:
    print("\nobj is None check no tha ->", type(e).__name__, "-", e)
print("Isi for every __get__ of pehli line: if obj is None: return self")

## 17. LeetCode — Design problems (descripthenrs/attribute-access nazarithese from)Spec of mutheniq: every notebook in LeetCode section. These chaar problems **class design** of are, andevery ek in hum specifically dekhenge of **who sa attribute-access or data-structure choice** faislakun is.### LC 146. LRU Cache — Medium🔗 https://leetcode.com/problems/lru-cache/**Bayan:** `LRUCache(capacity)` create that `get(key)` and `put(key, value)` support doe, both**O(1)** me. Capacity bhar jathese then **least recently used** key nikal do.**Approach (OOP nazariya):**- Needed: O(1) lookup **and** O(1) recency reorder. Ek only structure both not give can → **do**structures jor do: `dict` (lookup) + **doubly linked list** (order).- `Node` class = linked list of element (`prev`, `next`, `key`, `val`). Here `__slots__` istemaldoenge — lakhon nodes par memory bachegi and access tez will be (section 15 of allaq).- Sentinel `head`/`tail` dummy nodes → edge cases (khali list, first/last element) end.- Python in shortcut: `OrderedDict.move_then_end` — both versions dikha rahe are.**Time:** `get` O(1), `put` O(1) · **Space:** O(capacity)

In [ ]:
class _Node:
    """Doubly linked list node — __slots__ from memory/speed behtar."""
    __slots__ = ("key", "val", "prev", "next")

    def __init__(self, key=None, val=None):
        self.key, self.val = key, val
        self.prev = self.next = None


class LRUCache:
    """dict + doubly linked list => O(1) get/put."""

    def __init__(self, capacity: int):
        if capacity <= 0:
            raise ValueError("capacity > 0 honi should")
        self.capacity = capacity
        self._map = {}                       # key -> _Node  (O(1) lookup)
# sentinels: head.next = most recent, tail.prev = least recent        self._head, self._tail = _Node(), _Node()
        self._head.next, self._tail.prev = self._tail, self._head

# --- linked list helpers ---    def _remove(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _add_front(self, node):
        node.prev, node.next = self._head, self._head.next
        self._head.next.prev = node
        self._head.next = node

    def get(self, key: int) -> int:
        node = self._map.get(key)
        if node is None:
            return -1
        self._remove(node)                   # recency update
        self._add_front(node)
        return node.val

    def put(self, key: int, value: int) -> None:
        node = self._map.get(key)
        if node is not None:                 # update + promote
            node.val = value
            self._remove(node)
            self._add_front(node)
            return
        if len(self._map) >= self.capacity:  # evict LRU (tail from pehla)
            lru = self._tail.prev
            self._remove(lru)
            del self._map[lru.key]
        node = _Node(key, value)
        self._map[key] = node
        self._add_front(node)

    def keys_mru_to_lru(self):
        out, cur = [], self._head.next
        while cur is not self._tail:
            out.append(cur.key)
            cur = cur.next
        return out

    def __len__(self):
        return len(self._map)

    def __contains__(self, key):             # `key in cache` — recency change no does
        return key in self._map

    def __repr__(self):
        return f"LRUCache(cap={self.capacity}, mru->lru={self.keys_mru_to_lru()})"


# LeetCode official examplec = LRUCache(2)
c.put(1, 1)
c.put(2, 2)
assert c.get(1) == 1          # 1 recent ban gaya
c.put(3, 3)                   # 2 evict hui
assert c.get(2) == -1
c.put(4, 4)                   # 1 evict hui
assert c.get(1) == -1
assert c.get(3) == 3
assert c.get(4) == 4
print("LC146 official example ->", c)

# Extra testsc2 = LRUCache(3)
for i in (1, 2, 3):
    c2.put(i, i * 10)
c2.get(1)                     # 1 to promote do
c2.put(4, 40)                 # all from purani (2) nikle
assert c2.get(2) == -1 and c2.get(1) == 10
assert len(c2) == 3 and 3 in c2
print("extra tests ->", c2)
print("update-existing test:", end=" ")
c2.put(3, 999)
assert c2.get(3) == 999 and len(c2) == 3
print("ok")
print("[pass] LC 146")

In [ ]:
# Wahi problem, OrderedDict with — less code, wahi complexity.from collections import OrderedDict

class LRUCacheOD:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self._od = OrderedDict()

    def get(self, key: int) -> int:
        if key not in self._od:
            return -1
        self._od.move_to_end(key)                 # O(1) recency update
        return self._od[key]

    def put(self, key: int, value: int) -> None:
        if key in self._od:
            self._od.move_to_end(key)
        self._od[key] = value
        if len(self._od) > self.capacity:
            self._od.popitem(last=False)          # O(1) all from purana nikalo


# both implementations to random ops par match doao (differential testing)import random
random.seed(42)
a, b = LRUCache(4), LRUCacheOD(4)
for _ in range(3000):
    k = random.randint(1, 8)
    if random.random() < 0.5:
        assert a.get(k) == b.get(k), f"mismatch get({k})"
    else:
        v = random.randint(1, 100)
        a.put(k, v); b.put(k, v)
print("3000 random ops par both implementations identical -> [pass]")
print("Sabaq: manual linked-list interview me maanga goes is; production me OrderedDict use do.")

### LC 155. min Stack — Easy🔗 https://leetcode.com/problems/min-stack/**Bayan:** Stack that `push`, `pop`, `thenp`, and `getMin()` — all **O(1)** in doe.**Approach (OOP nazariya):**- Naive: `getMin()` in `min(stack)` → O(n). will run no.- Trick: **invariant** haveo — every element with it time of minimum also store do.Yani stack in `(value, min_so_far)` tuples. Push par `min(value, current_min)`.- Here `property` of acha use: `min_value` to computed read-only property create — API saaf,and inside from O(1).**Time:** all operations O(1) · **Space:** O(n)

In [ ]:
class MinStack:
    """(value, min_so_far) pairs => getMin O(1)."""

    __slots__ = ("_stack",)

    def __init__(self):
        self._stack = []          # list of (value, min_so_far)

    def push(self, val: int) -> None:
        current_min = val if not self._stack else min(val, self._stack[-1][1])
        self._stack.append((val, current_min))

    def pop(self) -> None:
        if not self._stack:
            raise IndexError("khali stack from pop")
        self._stack.pop()

    def top(self) -> int:
        if not self._stack:
            raise IndexError("khali stack of top")
        return self._stack[-1][0]

    def getMin(self) -> int:
        if not self._stack:
            raise IndexError("khali stack of min")
        return self._stack[-1][1]

# Pythonic API (property = data descripthenr, read-only)    @property
    def min_value(self):
        """getMin() of Pythonic chehra — read-only computed."""
        return self.getMin()

    def __len__(self):
        return len(self._stack)

    def __bool__(self):
        return bool(self._stack)

    def __repr__(self):
        return f"MinStack(values={[v for v, _ in self._stack]}, min={self.getMin() if self else None})"


# LeetCode official examples = MinStack()
s.push(-2); s.push(0); s.push(-3)
assert s.getMin() == -3
s.pop()
assert s.top() == 0
assert s.getMin() == -2
print("LC155 official ->", s)

# Duplicates and min propertys2 = MinStack()
for v in [5, 5, 3, 3, 7]:
    s2.push(v)
assert s2.min_value == 3
s2.pop(); s2.pop()
assert s2.min_value == 3, "ek 3 abhi baqi is"
s2.pop()
assert s2.min_value == 5
print("duplicates test ok ->", s2)

# property read-only istry:
    s2.min_value = 100
except AttributeError as e:
    print("min_value read-only ->", type(e).__name__)

for fn in (lambda: MinStack().pop(), lambda: MinStack().getMin(), lambda: MinStack().top()):
    try:
        fn()
    except IndexError:
        pass
    else:
        raise AssertionError("khali stack par error hona should tha")
print("[pass] LC 155")

### LC 706. Design HashMap — Easy🔗 https://leetcode.com/problems/design-hashmap/**Bayan:** Bina builtin dict/map of apna HashMap create: `put(key, value)`, `get(key)` (na mile then-1), `remove(key)`.**Approach (OOP nazariya):**- **Separate chaining**: buckets of list; every bucket ek list of `[key, value]`.- `hash(key) % num_buckets` from bucket chuno. Prime bucket count (2069) clustering less does.- Load facthenr increase jathese then **resize** (rehash) — yahi dict of amortized O(1) of raaz is.- Descripthenr connection: `obj.__dict__` itself such a only hash map is; `obj.x` = hash lookup + attributeprotocol. Isi lithese `__slots__` (array offset) `__dict__` from tez is.**Time:** average O(1), worst O(n) per op · **Space:** O(n)

In [ ]:
class MyHashMap:
    """Separate chaining + dynamic resize."""

    INITIAL_BUCKETS = 8
    MAX_LOAD = 0.75

    def __init__(self):
        self._buckets = [[] for _ in range(self.INITIAL_BUCKETS)]
        self._size = 0

    def _index(self, key):
        return hash(key) % len(self._buckets)

    def _resize(self):
        old = self._buckets
        self._buckets = [[] for _ in range(len(old) * 2)]
        self._size = 0
        for bucket in old:
            for k, v in bucket:
                self.put(k, v)                       # rehash

    def put(self, key: int, value: int) -> None:
        bucket = self._buckets[self._index(key)]
        for pair in bucket:                          # mojood key -> update
            if pair[0] == key:
                pair[1] = value
                return
        bucket.append([key, value])
        self._size += 1
        if self._size / len(self._buckets) > self.MAX_LOAD:
            self._resize()

    def get(self, key: int) -> int:
        for k, v in self._buckets[self._index(key)]:
            if k == key:
                return v
        return -1

    def remove(self, key: int) -> None:
        bucket = self._buckets[self._index(key)]
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self._size -= 1
                return

    def __len__(self):
        return self._size

    def __contains__(self, key):
        return self.get(key) != -1

    def stats(self):
        lengths = [len(b) for b in self._buckets]
        return {"buckets": len(self._buckets), "size": self._size,
                "longest_chain": max(lengths), "empty_buckets": lengths.count(0)}


# LeetCode official examplem = MyHashMap()
m.put(1, 1)
m.put(2, 2)
assert m.get(1) == 1
assert m.get(3) == -1
m.put(2, 1)                 # update
assert m.get(2) == 1
m.remove(2)
assert m.get(2) == -1
print("LC706 official example -> ok")

# Resize + bara load test, and builtin dict from muqnowlam2 = MyHashMap()
reference = {}
for i in range(1000):
    m2.put(i, i * i)
    reference[i] = i * i
assert all(m2.get(k) == v for k, v in reference.items())
print("1000 inserts ->", m2.stats())

for k in range(0, 1000, 3):
    m2.remove(k)
    del reference[k]
assert all(m2.get(k) == v for k, v in reference.items())
assert all(m2.get(k) == -1 for k in range(0, 1000, 3))
assert len(m2) == len(reference)
print("deletions of baad size match ->", len(m2))
print("0 in m2? ->", 0 in m2, "| 1 in m2? ->", 1 in m2)
print("[pass] LC 706")

### LC 380. Insert Delete GetRandom O(1) — Medium🔗 https://leetcode.com/problems/insert-delete-getrandom-o1/**Bayan:** `insert(val)`, `remove(val)`, `getRandom()` — teeno **average O(1)**. `getRandom` everymojood element to barnowar probnowility give.**Approach (OOP nazariya):**- `getRandom` O(1) for **contiguous array** needed (`random.choice` index from). But array fromremove O(n) is.- Trick: **swap-with-last** — that element hatana is use last element from largel do, then `pop()`.O(1) be gaya.- Position track doing for `dict: val -> index`. Yani then **do structures of combo**(LC 146 that only usool).- Design lesson: "O(1) random access" + "O(1) delete" ek structure from not milte → **compose** do.**Time:** teeno average O(1) · **Space:** O(n)

In [ ]:
import random

class RandomizedSet:
    """list (random access) + dict val->index (O(1) locate) + swap-with-last delete."""

    def __init__(self):
        self._items = []          # dense array
        self._pos = {}            # val -> index in _items

    def insert(self, val: int) -> bool:
        if val in self._pos:
            return False
        self._pos[val] = len(self._items)
        self._items.append(val)
        return True

    def remove(self, val: int) -> bool:
        idx = self._pos.get(val)
        if idx is None:
            return False
        last = self._items[-1]
# last element to hatne that jagah par le aao        self._items[idx] = last
        self._pos[last] = idx
        self._items.pop()
        del self._pos[val]
        return True

    def getRandom(self) -> int:
        if not self._items:
            raise IndexError("khali set")
        return random.choice(self._items)     # O(1)

    def __len__(self):
        return len(self._items)

    def __contains__(self, val):
        return val in self._pos

    def _check_invariant(self):
        """every val of index theek is? (testing of for)"""
        assert len(self._items) == len(self._pos)
        for v, i in self._pos.items():
            assert self._items[i] == v, f"invariant tuta: {v} index {i} par no"
        return True


# LeetCode official examplers = RandomizedSet()
assert rs.insert(1) is True
assert rs.remove(2) is False
assert rs.insert(2) is True
assert rs.getRandom() in (1, 2)
assert rs.remove(1) is True
assert rs.insert(2) is False
assert rs.getRandom() == 2
print("LC380 official example -> ok")

# Invariant stress testrandom.seed(7)
rs2 = RandomizedSet()
mirror = set()
for _ in range(4000):
    v = random.randint(1, 60)
    if random.random() < 0.55:
        assert rs2.insert(v) == (v not in mirror)
        mirror.add(v)
    else:
        assert rs2.remove(v) == (v in mirror)
        mirror.discard(v)
    rs2._check_invariant()
assert len(rs2) == len(mirror) and all(v in rs2 for v in mirror)
print(f"4000 random ops, invariant qaim, size={len(rs2)}")

# Distribution fairness (uniform?)rs3 = RandomizedSet()
for v in range(5):
    rs3.insert(v)
random.seed(1)
counts = {}
for _ in range(50_000):
    k = rs3.getRandom()
    counts[k] = counts.get(k, 0) + 1
print("distribution ->", dict(sorted(counts.items())))
assert all(8_500 < c < 11_500 for c in counts.values()), "kaafi uniform hona should"
print("[pass] LC 380")

## 18. Khud do — ExercisesKoshish itself do, then below solution cell chala do match do.1. **`ChoiceField`** descripthenr: only given options in from value accept doe, and `options` toread-only property from expose doe.2. **`ReadOnlyAfterSet`** descripthenr: pehli time set be jathese, uske after any change not(write-once field).3. **`Alias`** non-data descripthenr: `full = Alias("name")` → `obj.full` = `obj.name`.4. **`CallCounter`** descripthenr: method par lagao, these count doe method kitni time chali (perinstance).5. **`ValidatedDict`**: `__getattr__` + `__setattr__` from dict jise attribute of way use do,but only allowed keys.

In [ ]:
# --- Solutions ---
# 1) ChoiceFieldclass ChoiceField:
    def __init__(self, *options):
        if not options:
            raise ValueError("kam az kam ek option do")
        self._options = tuple(options)

    @property
    def options(self):                # read-only
        return self._options

    def __set_name__(self, owner, name):
        self.name = name
        self.storage = f"_choice_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.storage, self._options[0])   # pehla = default

    def __set__(self, obj, value):
        if value not in self._options:
            raise ValueError(f"{self.name}: {value!r} allowed no; options: {self._options}")
        obj.__dict__[self.storage] = value


# 2) ReadOnlyAfterSetclass ReadOnlyAfterSet:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage = f"_ro_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.storage not in obj.__dict__:
            raise AttributeError(f"{self.name} abhi set no hua")
        return obj.__dict__[self.storage]

    def __set__(self, obj, value):
        if self.storage in obj.__dict__:
            raise AttributeError(f"{self.name} write-once is; ab {self.name} badal no can")
        obj.__dict__[self.storage] = value


# 3) Alias (non-data: only __get__)class Alias:
    def __init__(self, target):
        self.target = target

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.target)


# 4) CallCounter (non-data descripthenr that bound wrapper gives is)import functools

class CallCounter:
    def __init__(self, func):
        self.func = func
        functools.update_wrapper(self, func)

    def __set_name__(self, owner, name):
        self.name = name
        self.counter_attr = f"_calls_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self

        @functools.wraps(self.func)
        def wrapper(*a, **kw):
            obj.__dict__[self.counter_attr] = obj.__dict__.get(self.counter_attr, 0) + 1
            return self.func(obj, *a, **kw)

        wrapper.call_count = obj.__dict__.get(self.counter_attr, 0)
        return wrapper

    @staticmethod
    def count_for(obj, method_name):
        return obj.__dict__.get(f"_calls_{method_name}", 0)


# 5) ValidatedDictclass ValidatedDict:
    ALLOWED = {"host", "port", "user"}

    def __init__(self, **kw):
        object.__setattr__(self, "_data", {})
        for k, v in kw.items():
            setattr(self, k, v)

    def __getattr__(self, name):
        try:
            return self._data[name]
        except KeyError:
            raise AttributeError(f"{name!r} set no is") from None

    def __setattr__(self, name, value):
        if name.startswith("_"):
            object.__setattr__(self, name, value)
            return
        if name not in self.ALLOWED:
            raise AttributeError(f"{name!r} allowed no; {sorted(self.ALLOWED)}")
        self._data[name] = value

    def __delattr__(self, name):
        self._data.pop(name, None)

    def __dir__(self):
        return sorted(set(list(super().__dir__()) + list(self._data)))

    def __repr__(self):
        return f"ValidatedDict({self._data})"


# ---- Tests ----class Order:
    status = ChoiceField("pending", "shipped", "delivered", "cancelled")
    order_id = ReadOnlyAfterSet()
    name = "Widget Order"
    label = Alias("name")

    def __init__(self, order_id):
        self.order_id = order_id

    @CallCounter
    def ship(self):
        self.status = "shipped"
        return "shipped!"


o = Order("ORD-1")
print("1) default status ->", o.status)
o.status = "shipped"
print("   set of baad    ->", o.status, "| options:", Order.status.options)
try:
    o.status = "teleported"
except ValueError as e:
    print("   [ok]", e)

print("\n2) order_id ->", o.order_id)
try:
    o.order_id = "ORD-2"
except AttributeError as e:
    print("   [ok]", e)

print("\n3) alias label ->", o.label, "(name from aaya)")

print("\n4) call counts:")
for _ in range(3):
    o.ship()
print("   ship() chali ->", CallCounter.count_for(o, "ship"), "dafa")
o2 = Order("ORD-9")
o2.ship()
print("   naye object par ->", CallCounter.count_for(o2, "ship"), "(per-instance)")

vd = ValidatedDict(host="localhost", port=8080)
print("\n5)", vd, "| vd.host ->", vd.host)
try:
    vd.hostt = "typo"
except AttributeError as e:
    print("   [ok]", e)
try:
    vd.user
except AttributeError as e:
    print("   [ok]", e)

assert o.status == "shipped" and Order.status.options[0] == "pending"
assert CallCounter.count_for(o, "ship") == 3 and CallCounter.count_for(o2, "ship") == 1
assert o.label == "Widget Order" and vd.port == 8080
print("\n[pass] saare exercises")

## 19. Interview Q&A (quick)**Q1. `obj.x` of lookup order?**metaclass path chhor do: `type(obj).__mro__` in **data descripthenr** → `obj.__dict__` →class attr / **non-data descripthenr** → `__getattr__` → `AttributeError`.**Q2. Data vs non-data descripthenr?**Data = `__set__`/`__delete__` mojood → instance dict from **oopar**. Non-data = only `__get__` →instance dict from **below**. `property` data is, `cached_property`/functions non-data.**Q3. `cached_property` how work doti is it?**Non-data descripthenr: pehli time compute do of result `obj.__dict__[name]` in daal deti is. Aglitime instance dict jeet goes → descripthenr absolutely runs only not → zero overhead. Invalidatedoing for dict from key delete do.**Q4. method to `self` who gives is it?**function ek non-data descripthenr is. `obj.m` → `function.__get__(obj, type(obj))` → bound method,that `obj` to first argument create gives is.**Q5. `__getattr__` vs `__getattribute__`?**`__getattribute__` every access par (default in complete algorithm); `__getattr__` only naalessi of after.Override doing for `__getattr__` safe and sasta.**Q6. `__set_name__` why aaya?**of descripthenr to apna attribute naam authenmatic mil jathese (first manually to give parta tha). class bodyof outside attach doing par itself not runs.**Q7. Descripthenr par per-instance data why not have can?**Descripthenr object poori class for **ek** only is. `obj.__dict__` or `WeakKeyDictionary` use do.**Q8. `property` when, descripthenr class when?**Ek attribute of logic → `property`. Wahi logic **kai** attributes/classes in reuse then do be(validation, typed fields, ORM) → descripthenr class.**Q9. `__slots__` descripthenrs with?**`__slots__` itself slot descripthenrs creates. Same naam par conflict is → descripthenr forseparate sthenrage naam (`_x`) haveo or weak dict.**Q10. `__getattr__` in `AttributeError` not rsuch do then?**`hasattr` hamesha `True`, and `copy`/`pickle` that `__deepcopy__`/`__reduce__`/`__getstate__` dhoondteare, that galat objects pa do bizarre errors dete are.

## 20. Summary & Cheat-sheet### Lookup order (yaad do)```obj.x:1. type(obj).__mro__ in data descripthenr?  -> desc.__get__(obj, type(obj))2. obj.__dict__["x"]?      -> wahi value3. class attr / non-data descripthenr?   -> value or desc.__get__(...)4. type(obj).__getattr__(obj, "x")   -> last mauqa5. AttributeError```### Descripthenr skelethenn (copy-paste)```pythonclass MyField:def __set_name__(self, owner, name):self.name = nameself.sthenrage = f"_{name}"def __get__(self, obj, objtype=None):if obj is None:return self      # class from accessreturn obj.__dict__[self.sthenrage]def __set__(self, obj, value):# validate...obj.__dict__[self.sthenrage] = value # PER-INSTANCEdef __delete__(self, obj):obj.__dict__.pop(self.sthenrage, None)```### Faislthenle| Needed | Istemal do ||---|---|| Ek computed/validated attribute | `property` || Wahi logic kai fields/classes in | descripthenr class || Mehngi value ek time compute | `functhenols.cached_property` || Dynamic/unknown attribute names | `__getattr__` || every assignment intercept | `__setattr__` || Binding rokna | `staticmethod` || Alternative constructor | `classmethod` || Memory + speed, fixed fields | `__slots__` || Field collect then do class banne par | `__init_subclass__` (NB2) |### Dunder cheat-sheet| Dunder | Trigger ||---|---|| `__getattribute__` | every `obj.x` || `__getattr__` | `obj.x` fail hone par || `__setattr__` | `obj.x = v` || `__delattr__` | `del obj.x` || `__dir__` | `dir(obj)` || `__get__/__set__/__delete__` | descripthenr access || `__set_name__` | class creation (ek dafa) |### Aage what**NB2 — Metaclasses & Dynamic OOP:** `__init_subclass__` (jo we have ORM in use done) of pooriwherei, `type()` from runtime class createna, class creation of lifecycle, plugin registry, and "whenmetaclass warranted is" (mostly no).---**Ththis notebook of thenpics cover hue:** attribute lookup algorithm ✓ · `__getattribute__` +recursion trap ✓ · `__getattr__` vs `__getattribute__` ✓ · `__setattr__`/`__delattr__`/`__dir__` ✓ ·descripthenr protocol ✓ · data vs non-data proof ✓ · function→method binding ✓ · `property` scratch from ✓ ·`classmethod`/`staticmethod` scratch from ✓ · TypedField/RangeField/Positive/NonEmpty/UnitField/AuditedField/LazyProperty/cached_property ✓ · slots + dataclass interplay ✓ · ORM Model ✓ ·pitfalls ✓ · performance ✓ · LeetCode 146/155/706/380 ✓